# Chapter 19 — The Full Training Loop on GPU

> Course: **llm.c — Zero to Hero**, Chapter 19 of ~20.
> Builds on every prior chapter (esp. Ch 8 CPU loop, Ch 17 mixed precision, Ch 18 AdamW + global norm).

This is the payoff chapter. You've written every piece — encoder, layernorm, matmul, attention, GELU, the fused classifier, AdamW, the global-norm reduction. Now we assemble them into the **one loop that actually trains a model**, and — crucially — we *run a complete training loop end to end on the GPU* so you watch a loss curve go down, not just read about one.

Two things make a real training loop more than "call forward, call backward, call AdamW":

1. **Gradient accumulation** — global batches are bigger than one GPU's memory, so we split each step into `K` micro-batches, accumulate gradients with `+=`, then take **one** optimizer step. We'll *prove numerically* that this equals one big batch.
2. **The per-step orchestration order** — zero grads → K×(forward, backward) → global grad-norm → clip → AdamW → cast to BF16. Get the order wrong and training silently diverges.

We'll build a small but **complete and runnable** training loop (a linear model, ~30 lines of CUDA reusing exactly the Ch 18 pipeline) so the orchestration is concrete, then trace where every transformer kernel slots into the real `train_gpt2.cu`.

### Learning objectives

By the end of this chapter you will:

- Explain and **demonstrate** gradient accumulation: K micro-batches `+=` into one buffer == one big batch.
- Run a full GPU training loop (forward → loss → backward → grad-norm → clip → AdamW) and watch loss drop.
- Trace `gpt2_forward` and `gpt2_backward_and_reduce` in `train_gpt2.cu`, naming every kernel.
- State the exact per-step order in `train_gpt2.cu`'s `main()` and why each step is where it is.


In [1]:
!mkdir -p course/ch19_build


## 1. Concept — Gradient Accumulation (the thing that makes it a *loop*)

Real GPT-2 training uses a **global batch** (e.g. 0.5M tokens) far larger than what fits on one GPU at once. The trick: pick a micro-batch `B×T` that *does* fit, and do `K = total_batch / (B*T)` forward+backward passes **without** running the optimizer, each one **adding** its gradients into the same `grads_memory`. After `K` micro-batches, run AdamW **once**.

Why this is exactly equal to one big batch: the loss is a **mean** over tokens. The gradient of a mean is the mean of the per-example gradients — a **sum**, rescaled by `1/total`. So if every micro-batch's backward pass already multiplies by `1/total` and we accumulate with `+=`, the K partial gradients sum to the full-batch gradient. That `1/total` is precisely this line in `train_gpt2.cu`:

```cpp
// gpt2_backward_and_reduce(), llmc backward
const float dloss = 1.0f / (float)(B * T * grad_accum_steps); // uniform average over ALL tokens
```

Let's not take that on faith. Below, one CUDA program computes the gradient of a linear model's MSE loss **two ways** — as one batch of `N` samples, and as `K` accumulated micro-batches of `N/K` — and checks they match to float precision.


**Concrete example — two micro-batches by hand.** Take 4 samples whose per-sample gradient contributions are `[2, 6, 4, 8]`, with a global batch `N = 4`. The full-batch gradient is the mean: `(2 + 6 + 4 + 8) / 4 = 5`. Now split into `K = 2` micro-batches of 2, each scaled by the **same** `1/N = 1/4`:

- micro-batch 0 sees `[2, 6]`:  contributes `(2 + 6)/4 = 2.0`  →  `grad += 2.0`
- micro-batch 1 sees `[4, 8]`:  contributes `(4 + 8)/4 = 3.0`  →  `grad += 3.0`

After both, `grad = 2.0 + 3.0 = 5.0` — identical to the full batch. The subtle part is the scale: each micro-batch divides by the **global** count `N`, not by its own size `per`. Use `1/per` instead and you'd get `4 + 6 = 10` — wrong by exactly `K×`. That is why the production `dloss` divides by `B * T * grad_accum_steps`, not by `B * T`.


In [2]:
%%writefile course/ch19_build/grad_accum.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

// Linear model p_n = dot(w, x_n).  Loss = (1/N) * sum_n 0.5*(p_n - t_n)^2.
// d Loss / d w_j = (1/N) * sum_n (p_n - t_n) * x_{n,j}.
// This kernel ADDS the contribution of samples [s0, s1) into grad[], scaled by inv_total = 1/N.
// One thread per feature j.
__global__ void accum_grad(float* grad, const float* X, const float* w, const float* t,
                           int N, int D, int s0, int s1, float inv_total) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= D) return;
    float g = 0.0f;
    for (int n = s0; n < s1; n++) {
        float p = 0.0f;
        for (int k = 0; k < D; k++) p += w[k] * X[n*D + k];   // prediction for sample n
        g += (p - t[n]) * X[n*D + j];
    }
    grad[j] += inv_total * g;     // '+=' is the whole point: accumulation
}

int main(void) {
    int N = 4096, D = 16, K = 8;          // N samples, D features, K micro-batches
    int per = N / K;                       // samples per micro-batch
    float *X = (float*)malloc(N*D*4), *w = (float*)malloc(D*4), *t = (float*)malloc(N*4);
    for (int n = 0; n < N; n++) {
        for (int k = 0; k < D; k++) X[n*D+k] = sinf(0.1f*(n+1)*(k+1));
        t[n] = cosf(0.07f*(n+1));
    }
    for (int k = 0; k < D; k++) w[k] = 0.05f * (k - 8);

    float *dX, *dw, *dt, *g_full, *g_accum;
    cudaMalloc(&dX, N*D*4); cudaMalloc(&dw, D*4); cudaMalloc(&dt, N*4);
    cudaMalloc(&g_full, D*4); cudaMalloc(&g_accum, D*4);
    cudaMemcpy(dX, X, N*D*4, cudaMemcpyHostToDevice);
    cudaMemcpy(dw, w, D*4,   cudaMemcpyHostToDevice);
    cudaMemcpy(dt, t, N*4,   cudaMemcpyHostToDevice);
    float inv_total = 1.0f / (float)N;

    // (A) one big batch: all N samples at once
    cudaMemset(g_full, 0, D*4);
    accum_grad<<<1, D>>>(g_full, dX, dw, dt, N, D, 0, N, inv_total);

    // (B) K micro-batches, each adding into the SAME buffer (zero once, before the loop)
    cudaMemset(g_accum, 0, D*4);
    for (int micro = 0; micro < K; micro++) {
        int s0 = micro * per, s1 = s0 + per;
        accum_grad<<<1, D>>>(g_accum, dX, dw, dt, N, D, s0, s1, inv_total);
    }

    float hf[16], ha[16];
    cudaMemcpy(hf, g_full,  D*4, cudaMemcpyDeviceToHost);
    cudaMemcpy(ha, g_accum, D*4, cudaMemcpyDeviceToHost);
    float maxdiff = 0.0f;
    for (int j = 0; j < D; j++) maxdiff = fmaxf(maxdiff, fabsf(hf[j]-ha[j]));
    printf("grad[0..3] full : %+.6f %+.6f %+.6f %+.6f\n", hf[0],hf[1],hf[2],hf[3]);
    printf("grad[0..3] accum: %+.6f %+.6f %+.6f %+.6f   (K=%d micro-batches)\n", ha[0],ha[1],ha[2],ha[3], K);
    printf("max |full - accumulated| = %.2e   -> %s\n", maxdiff, maxdiff < 1e-5 ? "PASS" : "FAIL");

    cudaFree(dX);cudaFree(dw);cudaFree(dt);cudaFree(g_full);cudaFree(g_accum);
    free(X);free(w);free(t);
    return 0;
}


Overwriting course/ch19_build/grad_accum.cu


In [3]:
!nvcc -O2 -o course/ch19_build/grad_accum course/ch19_build/grad_accum.cu && ./course/ch19_build/grad_accum


grad[0..3] full : -0.208683 -0.176933 -0.151221 -0.126181
grad[0..3] accum: -0.208683 -0.176933 -0.151221 -0.126181   (K=8 micro-batches)
max |full - accumulated| = 1.49e-07   -> PASS


Identical to ~`1e-7`. **8 micro-batches accumulated with `+=` gave the exact same gradient as one batch of 4096** — the only difference is peak memory (one micro-batch of activations at a time instead of all 4096). This is *the* mechanism that lets an 8-GPU node train a model whose global batch never fits in memory at once. Two things to lock in:

- **Zero the gradient buffer once, before the micro-batch loop** — not inside it. (That's `gpt2_zero_grad` before the `for micro_step` loop.)
- The `1/total` scale lives **inside the backward** so each `+=` already contributes its averaged share.


## 2. Demo — A Complete Training Loop That Actually Trains

The transformer kernels are big, but the *loop* around them is small and identical for any model. Here it is, end to end, on the GPU — fitting a linear model `w` to data generated from a known `w_true`, using **exactly the production per-step pipeline**:

```
for each step:
    zero_grad                         # clear the gradient buffer
    grad = backward(forward(x))       # one micro-batch here (K=1) for clarity
    grad_norm = sqrt(sum(grad^2))     # Ch 18 global-norm reduction
    scale = (grad_norm > 1) ? 1/grad_norm : 1   # Ch 18 gradient clipping
    AdamW(w, m, v, grad * scale)      # Ch 17/18 optimizer, betas (0.9, 0.95)
```

Same control flow as `train_gpt2.cu`'s `main()` — only the model is a single matmul instead of 12 transformer blocks. Watch the loss fall.


In [ ]:
%%writefile course/ch19_build/train_loop.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

__device__ float warpReduceSum(float v){ for(int o=16;o>0;o/=2) v+=__shfl_down_sync(0xffffffff,v,o); return v; }
__device__ float blockReduceSum(float v){
    __shared__ float ws[32];
    int lane=threadIdx.x&31, wid=threadIdx.x>>5, nw=(blockDim.x+31)/32;
    v=warpReduceSum(v); if(lane==0) ws[wid]=v; __syncthreads();
    v=(threadIdx.x<nw)?ws[threadIdx.x]:0.0f; if(wid==0) v=warpReduceSum(v); return v;
}

// FORWARD + BACKWARD fused: grad[j] = (1/N) sum_n (p_n - t_n) x_nj.
__global__ void grad_kernel(float* grad, const float* X, const float* w, const float* t, int N, int D) {
    int j = blockIdx.x*blockDim.x + threadIdx.x;
    if (j >= D) return;
    float g = 0.0f;
    for (int n = 0; n < N; n++) {
        float p = 0.0f;
        for (int k = 0; k < D; k++) p += w[k]*X[n*D+k];
        g += (p - t[n]) * X[n*D+j];
    }
    grad[j] = g / (float)N;
}
// LOSS: (1/N) sum_n 0.5 (p_n - t_n)^2, single-block reduction.
__global__ void loss_kernel(float* out, const float* X, const float* w, const float* t, int N, int D) {
    float local = 0.0f;
    for (int n = threadIdx.x; n < N; n += blockDim.x) {
        float p = 0.0f;
        for (int k = 0; k < D; k++) p += w[k]*X[n*D+k];
        float e = p - t[n];
        local += 0.5f * e * e;
    }
    float s = blockReduceSum(local);
    if (threadIdx.x == 0) out[0] = s / (float)N;
}
// GLOBAL NORM^2 of the gradient (Ch 18), single block.
__global__ void normsq_kernel(float* out, const float* grad, int D) {
    float local = 0.0f;
    for (int j = threadIdx.x; j < D; j += blockDim.x) local += grad[j]*grad[j];
    float s = blockReduceSum(local);
    if (threadIdx.x == 0) out[0] = s;
}
// AdamW with the clip scale folded in (Ch 17/18). grad_scale comes from clipping.
__global__ void adamw_kernel(float* w, float* m, float* v, const float* grad, int D,
                             float lr, float b1, float b2, float eps, float wd, float grad_scale, int t) {
    int j = blockIdx.x*blockDim.x + threadIdx.x;
    if (j >= D) return;
    float g = grad[j] * grad_scale;                  // clip applied here
    float mn = b1*m[j] + (1-b1)*g;
    float vn = b2*v[j] + (1-b2)*g*g;
    m[j]=mn; v[j]=vn;
    float mh = mn/(1-powf(b1,t)), vh = vn/(1-powf(b2,t));
    w[j] -= lr * (mh/(sqrtf(vh)+eps) + wd*w[j]);
}

int main(void) {
    int N = 1024, D = 32, steps = 200;
    float *X=(float*)malloc(N*D*4), *t=(float*)malloc(N*4), w_true[32];
    for (int k=0;k<D;k++) w_true[k] = 0.3f*sinf(0.5f*k) + 0.1f*k;       // the target weights
    for (int n=0;n<N;n++){
        float p=0; for(int k=0;k<D;k++){ X[n*D+k]=sinf(0.13f*(n+1)*(k+3)); p+=w_true[k]*X[n*D+k]; }
        t[n]=p;                                                          // noiseless: loss should -> ~0
    }
    float *dX,*dt,*dw,*dm,*dv,*dgrad,*dloss,*dnsq;
    cudaMalloc(&dX,N*D*4);cudaMalloc(&dt,N*4);cudaMalloc(&dw,D*4);cudaMalloc(&dm,D*4);
    cudaMalloc(&dv,D*4);cudaMalloc(&dgrad,D*4);cudaMalloc(&dloss,4);cudaMalloc(&dnsq,4);
    cudaMemcpy(dX,X,N*D*4,cudaMemcpyHostToDevice); cudaMemcpy(dt,t,N*4,cudaMemcpyHostToDevice);
    cudaMemset(dw,0,D*4); cudaMemset(dm,0,D*4); cudaMemset(dv,0,D*4);   // start from w=0

    float lr=0.05f, b1=0.9f, b2=0.95f, eps=1e-8f, wd=0.0f, max_norm=1.0f;
    printf("step    loss        grad_norm  scale\n");
    for (int step=1; step<=steps; step++) {
        // --- the full per-step pipeline, same order as train_gpt2.cu main() ---
        grad_kernel<<<1, D>>>(dgrad, dX, dw, dt, N, D);          // forward+backward
        loss_kernel<<<1, 256>>>(dloss, dX, dw, dt, N, D);        // (diagnostic) loss
        normsq_kernel<<<1, 256>>>(dnsq, dgrad, D);               // Ch 18 global norm^2
        float loss, nsq; cudaMemcpy(&loss,dloss,4,cudaMemcpyDeviceToHost);
        cudaMemcpy(&nsq,dnsq,4,cudaMemcpyDeviceToHost);
        float gnorm = sqrtf(nsq);
        float scale = (gnorm > max_norm) ? max_norm/gnorm : 1.0f;  // Ch 18 clip
        adamw_kernel<<<1, D>>>(dw,dm,dv,dgrad,D, lr,b1,b2,eps,wd, scale, step);
        if (step==1 || step%20==0)
            printf("%4d   %9.6f   %8.4f   %.3f\n", step, loss, gnorm, scale);
    }
    // recover w, report fit error vs w_true
    float w[32]; cudaMemcpy(w,dw,D*4,cudaMemcpyDeviceToHost);
    float werr=0; for(int k=0;k<D;k++) werr=fmaxf(werr,fabsf(w[k]-w_true[k]));
    printf("\nmax |w_learned - w_true| = %.4f   -> %s\n", werr, werr < 0.05f ? "PASS (converged)" : "still training");
    cudaFree(dX);cudaFree(dt);cudaFree(dw);cudaFree(dm);cudaFree(dv);cudaFree(dgrad);cudaFree(dloss);cudaFree(dnsq);
    free(X);free(t);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch19_build/train_loop course/ch19_build/train_loop.cu && ./course/ch19_build/train_loop


The loss falls from ~its initial value toward zero and `w` converges to `w_true`. Notice the **`scale` column**: in the first few steps the gradient is large so `scale < 1` (clipping is active, holding the step bounded); once the gradient shrinks, `scale = 1` and AdamW runs unclipped. That is the entire behavior of gradient clipping in real training — early steps are the volatile ones.

This loop is structurally `train_gpt2.cu`. Swap the single `grad_kernel` for `gpt2_forward` + `gpt2_backward_and_reduce`, swap one feature-vector for 124M parameters, wrap the inner forward/backward in a `K`-micro-batch loop, and you have the real thing — which we trace next.


## 3. The Real `gpt2_forward` in `train_gpt2.cu`

The CUDA forward (`gpt2_forward`, ~line 646) mirrors the CPU forward (Ch 8) step for step — the difference is *which* kernel runs each layer:

| Step | CPU `train_gpt2.c` | GPU `train_gpt2.cu` |
|---|---|---|
| Embedding | `encoder_forward` | `encoder_forward_kernel3` (Ch 10) |
| LayerNorm | `layernorm_forward` | `layernorm_forward_kernel6` — warp-per-row + Packed128 (Ch 13) |
| QKV linear | `matmul_forward` | `matmul_forward_cublaslt` — cuBLAS (Ch 14) |
| Attention | `attention_forward` | `softmax_forward_kernel5` + cuBLAS, **or** `attention_forward_cudnn` Flash Attention (Ch 16) |
| Attn-out proj + residual | two calls | `matmul_forward_cublaslt` with `accumulate=true` — adds the residual *inside* the matmul |
| Residual + LayerNorm | two calls | **`fused_residual_forward_kernel`** (Ch 15) |
| FFN up + GELU | two calls | `matmul_forward_cublaslt` with `EPILOGUE_GELU_AUX_BIAS` (Ch 14) |
| FFN down + residual | two calls | `matmul_forward_cublaslt` with `accumulate=true` |
| Logits + loss | `softmax` + `crossentropy` | **`fused_classifier`** — softmax+CE+`dlogits` in one (Ch 7, 15) |

The real signature (note: **no `targets` argument** — targets enter in the *backward*, and the loss is computed by the fused classifier):

```cpp
void gpt2_forward(GPT2 *model, const int* inputs, size_t B, size_t T) {
    encoder_forward(acts.encoded, model->inputs, params.wte, params.wpe, B, T, C);
    for (int l = 0; l < L; l++) {
        layernorm_forward(l_ln1, ..., l_ln1w, l_ln1b, B, T, C);
        matmul_forward_cublaslt(l_qkvr, l_ln1, l_qkvw, l_qkvb, B, T, C, 3*C);
        attention_forward(l_atty, l_qkvr, l_att, B, T, C, NH);
        matmul_forward_cublaslt(l_residual2, l_atty, l_attprojw, l_attprojb, ...,
                                /*accumulate=*/true, ...);     // proj + residual fused
        fused_residual_forward(...);                            // residual + LN
        matmul_forward_cublaslt(l_fch_gelu, l_ln2, l_fcw, l_fcb, ..., /*gelu epilogue*/ );
        matmul_forward_cublaslt(scratch, l_fch_gelu, l_fcprojw, ..., /*accumulate=*/true);
    }
    fused_residual_forward(... final LN ...);
    matmul_forward_cublaslt(acts.output, acts.lnf, params.wte, NULL, B, T, C, Vp);  // unembed
    // loss + dlogits produced later by fused_classifier in the backward entry
}
```

Two fusions worth re-noticing, both from Ch 15: `accumulate=true` folds the **residual add into the projection matmul** (`out = A@B + bias + residual`), and the **classifier fuses softmax+CE+`dlogits`** so the giant `(B,T,Vp)` probabilities tensor is never materialized.


> **New API vocabulary — cuBLASLt epilogues.** The table keeps naming flags like `EPILOGUE_GELU_AUX_BIAS` and `accumulate=true`. An *epilogue* is an extra elementwise step cuBLASLt runs **inside** the matmul, on-chip, before the result is ever written back to memory — so a plain `out = matmul(A, B)` becomes, in **one** kernel launch:
>
> ```c
> out = activation( matmul(A, B) + bias [+ residual] )
> ```
>
> `accumulate=true` adds an existing buffer (the residual) into the product. `EPILOGUE_GELU_AUX_BIAS` adds the bias, applies GELU, and stashes the pre-GELU values (the "AUX" tensor) that the backward will need. This is the same Ch 14/15 fusion idea you hand-wrote earlier — here it's just a **library flag** instead of a custom kernel, so cuBLAS does the fusion for you.


## 4. The Backward — `gpt2_backward_and_reduce`

`gpt2_backward_and_reduce` (~line 788) does **the chain rule in reverse, gradient accumulation, and (multi-GPU) the NCCL reduce** in one function. Its signature carries the accumulation bookkeeping you saw in Section 1:

```cpp
void gpt2_backward_and_reduce(GPT2 *model, int* inputs, const int* targets,
                              int grad_accum_steps, int micro_step) {
    bool last_step = micro_step == grad_accum_steps - 1;
    if (micro_step == 0) { /* zero the per-layer grad-activation scratch */ }

    // dloss scales EVERY gradient by 1/(B*T*grad_accum_steps) -> averaged over all tokens
    const float dloss = 1.0f / (float)(B * T * grad_accum_steps);
    fused_classifier(acts.output, acts.losses, dloss, targets, B, T, V, Vp);  // makes dlogits

    matmul_backward(..., grads.wte, ..., acts.output, ...);   // through the unembedding
    layernorm_backward(..., grads.lnfw, grads.lnfb, ...);     // final LN

    for (int l = L-1; l >= 0; l--) {                          // layers, REVERSE
        matmul_backward(...);     // FFN down  (DGELU fused via EPILOGUE_DGELU_BGRADB)
        matmul_backward(...);     // FFN up
        layernorm_backward(...);  // LN2
        residual_backward(...);   // residual gradient split (Ch 5)
        matmul_backward(...);     // attention output projection
        attention_backward(...);  // cudnn or hand-rolled (Ch 16)
        matmul_backward(...);     // QKV
        layernorm_backward(...);  // LN1
        residual_backward(...);   // first residual split
    }
    encoder_backward(grads.wte, grads.wpe, ..., inputs, B, T, C);  // embeddings

    if (last_step && multi_gpu) {                             // only after the K-th micro-batch
        multi_gpu_async_reduce_gradient(grads, ...);          // Ch 20: all-reduce / reduce-scatter
    }
}
```

Two subtleties that trip people up:

- **`fused_classifier` is called once, at the *top* of the backward** — it already produced `dlogits` (it depends only on `probs` and `targets`, both available in the forward). So the backward "starts" right at the unembedding matmul. That's the Ch 7 trick paying off.
- **The NCCL reduce only fires on the `last_step`** (`micro_step == grad_accum_steps - 1`). Communication happens **once per optimizer step**, not once per micro-batch — you accumulate K local gradients first, *then* sync. That's why Section 1's accumulation and Chapter 20's communication compose cleanly.


## 5. The Per-Step Order in `main()`

Here is the actual outer loop from `train_gpt2.cu` (~line 1828), lightly trimmed. This is the order your Section-2 demo imitated — now with accumulation and clipping wired in:

```cpp
for (int micro_step = 0; micro_step < grad_accum_steps; micro_step++) {
    dataloader_next_batch(&train_loader);                       // next micro-batch shard
    gpt2_forward(&model, train_loader.inputs, B, T);            // forward (no targets!)
    gpt2_backward_and_reduce(&model, train_loader.inputs,
                             train_loader.targets, grad_accum_steps, micro_step);  // += grads
}
float grad_norm = gpt2_calculate_grad_norm(&model, &multi_gpu_config);  // Ch 18, two-stage
float grad_clip = 1.0f;
float grad_scale = (grad_norm > grad_clip) ? grad_clip / grad_norm : 1.0f;  // Ch 18 clip
gpt2_update(&model, step_lr, 0.9f, 0.95f, 1e-8f, weight_decay,
            grad_scale, step+1, &multi_gpu_config);             // Ch 17/18 AdamW (+cast)
```

Read off the order — it is not arbitrary:

1. **K micro-batches of forward+backward** accumulate into `grads_memory`. No optimizer yet.
2. **Grad norm** is computed over the *fully accumulated* gradient (Ch 18's two-stage reduction).
3. **Clip scale** = `min(1, max_norm/grad_norm)` — computed *before* AdamW so it can be folded into the update.
4. **One `gpt2_update`**: AdamW in FP32 with `grad_scale` applied, then cast master FP32 → working BF16 (Ch 17).

Note the real betas: **`(0.9, 0.95)`**, not Adam's textbook `(0.9, 0.999)` — GPT-2 uses a lower `beta2`. And clipping uses `max_norm = 1.0`. Your Section-2 loop used these exact values.


The order, drawn out — `zero_grad` once, then `K` forward/backward passes accumulating into the same buffer, then a single norm → clip → update at the end:

```mermaid
flowchart LR
  Z["zero_grad (once)"] --> F
  subgraph MB["repeat K = grad_accum_steps times"]
    F["forward"] --> B["backward<br/>grads += (1/total) times dLoss"]
  end
  MB --> N["grad_norm<br/>two-stage reduction (Ch 18)"]
  N --> C["grad_scale =<br/>min(1, 1/grad_norm) (Ch 18)"]
  C --> U["gpt2_update<br/>AdamW(FP32) then cast to BF16 (Ch 17)"]
  U -->|next step| Z
```

The two things that must be *outside* the micro-batch loop: zeroing the gradient buffer (before) and the optimizer step (after). Everything between the `zero_grad` and the `grad_norm` is just `+=` into the same buffer.


## 6. Build (and optionally run) the Real `train_gpt2cu`

You can compile the production binary right now. On a single-GPU box without NCCL installed, the default build fails on `#include <nccl.h>` (pulled in by `llmc/zero.cuh`) — pass `NO_MULTI_GPU=1` for the single-GPU path:

```bash
make train_gpt2cu NO_MULTI_GPU=1     # single-GPU BF16 build (what we do below)
make train_gpt2cu USE_CUDNN=1        # add cuDNN Flash Attention (needs cuDNN + NCCL)
make train_gpt2fp32cu                # FP32 build (slower, simpler to debug)
```


In [ ]:
# Build train_gpt2cu (single-GPU path). Takes ~30-60s; NVCC compiles the whole training file.
import subprocess, os
r = subprocess.run(["make", "train_gpt2cu", "NO_MULTI_GPU=1"], capture_output=True, text=True)
print(r.stdout[-600:] if len(r.stdout) > 600 else r.stdout)
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-1500:])
else:
    print("BUILD OK")
    if os.path.exists("train_gpt2cu"):
        print(f"  binary: train_gpt2cu  ({os.path.getsize('train_gpt2cu')/1024:.1f} KB)")


If the build succeeded you have the real trainer. To actually train GPT-2 124M (optional — completing the course doesn't require it):

```bash
./dev/download_starter_pack.sh     # one-time ~520 MB: weights, tokenizer, debug state, tinyshakespeare
./train_gpt2cu                      # loss prints and goes down; every kernel is one you've studied
```

And to **prove the whole thing matches PyTorch bit-for-bit**, `test_gpt2.cu` loads the same `gpt2_124M_debug_state.bin` that `train_gpt2.py` writes, runs forward+backward+10 AdamW steps, and checks every intermediate tensor:

```bash
make test_gpt2cu NO_MULTI_GPU=1 && ./test_gpt2cu
# LOSS OK: 5.270 5.270 | dwte: TENSOR OK, maxdiff = 1.86e-04 | ... | step 0: loss 5.270 ... step 1: loss 4.06 ...
```

Seeing all `OK` and the loss decreasing is the final certificate: every line you've read produces the same numbers as PyTorch.


## 7. The Mental Map

The full GPU stack, outermost loop inward — every `Ch X` is something you've already built:

```
main()                                         (train_gpt2.cu)
└─ for step in 1..num_iterations:
   ├─ gpt2_zero_grad                            (clear grads — Section 1)
   ├─ for micro_step in 0..grad_accum_steps:    (gradient accumulation — Section 1)
   │  ├─ dataloader_next_batch                  (llmc/dataloader.h)
   │  ├─ gpt2_forward
   │  │  ├─ encoder_forward                      (Ch 10)
   │  │  ├─ for layer: layernorm / matmul_cublaslt / attention / fused_residual   (Ch 13,14,16,15)
   │  │  └─ fused_classifier (forward + dlogits) (Ch 7, 15)
   │  └─ gpt2_backward_and_reduce
   │     ├─ matmul_backward / layernorm_backward / attention_backward / encoder_backward (Ch 14,13,16,10)
   │     └─ multi_gpu reduce (last micro_step)   (Ch 20)
   ├─ gpt2_calculate_grad_norm                   (two-stage reduction — Ch 18)
   ├─ grad_scale = min(1, 1/grad_norm)           (clip — Ch 18)
   └─ gpt2_update: AdamW(FP32) + cast→BF16        (Ch 17, 18)
```


## 8. Exercise — Accumulate a Gradient Yourself

Below, the micro-batch loop is incomplete. Fill in the TODOs so that `K` micro-batches accumulate to the same gradient as one full batch. This is the core of every large-scale training loop. The asserts check `accum == full`.


In [ ]:
%%writefile course/ch19_build/exercise1.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

// grad[j] += inv_total * sum_{n in [s0,s1)} (dot(w,x_n) - t_n) * x_nj
__global__ void accum_grad(float* grad, const float* X, const float* w, const float* t,
                           int N, int D, int s0, int s1, float inv_total) {
    int j = blockIdx.x*blockDim.x + threadIdx.x;
    if (j >= D) return;
    float g = 0.0f;
    for (int n = s0; n < s1; n++) {
        float p = 0.0f; for (int k=0;k<D;k++) p += w[k]*X[n*D+k];
        g += (p - t[n]) * X[n*D+j];
    }
    grad[j] += inv_total * g;
}

int main(void) {
    int N = 2048, D = 8, K = 4, per = N/K;
    float *X=(float*)malloc(N*D*4), *w=(float*)malloc(D*4), *t=(float*)malloc(N*4);
    for(int n=0;n<N;n++){ for(int k=0;k<D;k++) X[n*D+k]=cosf(0.2f*(n+1)*(k+1)); t[n]=sinf(0.05f*(n+1)); }
    for(int k=0;k<D;k++) w[k]=0.1f*(k-4);
    float *dX,*dw,*dt,*g_full,*g_accum;
    cudaMalloc(&dX,N*D*4);cudaMalloc(&dw,D*4);cudaMalloc(&dt,N*4);cudaMalloc(&g_full,D*4);cudaMalloc(&g_accum,D*4);
    cudaMemcpy(dX,X,N*D*4,cudaMemcpyHostToDevice);cudaMemcpy(dw,w,D*4,cudaMemcpyHostToDevice);cudaMemcpy(dt,t,N*4,cudaMemcpyHostToDevice);
    float inv_total = 1.0f/(float)N;

    // reference: one big batch
    cudaMemset(g_full,0,D*4);
    accum_grad<<<1,D>>>(g_full, dX, dw, dt, N, D, 0, N, inv_total);

    // TODO 1: zero the accumulation buffer ONCE, before the loop (cudaMemset g_accum)
    // TODO 2: loop micro = 0..K-1, launch accum_grad on samples [micro*per, micro*per+per)
    //         passing the SAME inv_total each time.
    // (write your code here)

    float hf[8], ha[8];
    cudaMemcpy(hf,g_full,D*4,cudaMemcpyDeviceToHost);
    cudaMemcpy(ha,g_accum,D*4,cudaMemcpyDeviceToHost);
    float md=0; for(int j=0;j<D;j++) md=fmaxf(md,fabsf(hf[j]-ha[j]));
    printf("max |full - accum| = %.2e   -> %s\n", md, md<1e-5 ? "PASS" : "FAIL");
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch19_build/exercise1 course/ch19_build/exercise1.cu && ./course/ch19_build/exercise1


### Solution

In [ ]:
%%writefile course/ch19_build/exercise1_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

__global__ void accum_grad(float* grad, const float* X, const float* w, const float* t,
                           int N, int D, int s0, int s1, float inv_total) {
    int j = blockIdx.x*blockDim.x + threadIdx.x;
    if (j >= D) return;
    float g = 0.0f;
    for (int n = s0; n < s1; n++) {
        float p = 0.0f; for (int k=0;k<D;k++) p += w[k]*X[n*D+k];
        g += (p - t[n]) * X[n*D+j];
    }
    grad[j] += inv_total * g;
}

int main(void) {
    int N = 2048, D = 8, K = 4, per = N/K;
    float *X=(float*)malloc(N*D*4), *w=(float*)malloc(D*4), *t=(float*)malloc(N*4);
    for(int n=0;n<N;n++){ for(int k=0;k<D;k++) X[n*D+k]=cosf(0.2f*(n+1)*(k+1)); t[n]=sinf(0.05f*(n+1)); }
    for(int k=0;k<D;k++) w[k]=0.1f*(k-4);
    float *dX,*dw,*dt,*g_full,*g_accum;
    cudaMalloc(&dX,N*D*4);cudaMalloc(&dw,D*4);cudaMalloc(&dt,N*4);cudaMalloc(&g_full,D*4);cudaMalloc(&g_accum,D*4);
    cudaMemcpy(dX,X,N*D*4,cudaMemcpyHostToDevice);cudaMemcpy(dw,w,D*4,cudaMemcpyHostToDevice);cudaMemcpy(dt,t,N*4,cudaMemcpyHostToDevice);
    float inv_total = 1.0f/(float)N;

    cudaMemset(g_full,0,D*4);
    accum_grad<<<1,D>>>(g_full, dX, dw, dt, N, D, 0, N, inv_total);

    cudaMemset(g_accum, 0, D*4);                       // TODO 1: zero ONCE before the loop
    for (int micro = 0; micro < K; micro++) {          // TODO 2: accumulate K micro-batches
        int s0 = micro*per, s1 = s0 + per;
        accum_grad<<<1,D>>>(g_accum, dX, dw, dt, N, D, s0, s1, inv_total);
    }

    float hf[8], ha[8];
    cudaMemcpy(hf,g_full,D*4,cudaMemcpyDeviceToHost);
    cudaMemcpy(ha,g_accum,D*4,cudaMemcpyDeviceToHost);
    float md=0; for(int j=0;j<D;j++) md=fmaxf(md,fabsf(hf[j]-ha[j]));
    printf("max |full - accum| = %.2e   -> %s\n", md, md<1e-5 ? "PASS" : "FAIL");
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch19_build/exercise1_sol course/ch19_build/exercise1_sol.cu && ./course/ch19_build/exercise1_sol


## Further Reading

**Source of truth**

- `train_gpt2.cu` in this repo — `gpt2_forward`, `gpt2_backward_and_reduce`, `gpt2_update`, and the `main()` training loop this chapter traces.
- `test_gpt2.cu` — the PyTorch cross-check that validates every tensor in the loop.
- The [llm.c repository](https://github.com/karpathy/llm.c) (Andrej Karpathy) — upstream README and the `dev/cuda` kernel-by-kernel history this course follows.

**Going deeper**

- [_How To Scale Your Model_](https://jax-ml.github.io/scaling-book/) — gradient accumulation and the batch/memory trade-off this chapter demonstrates.
- [CUDA C++ Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html) — the optimization checklist every kernel in the loop satisfies.


## Recap

You now know — and have **run**:

- **Gradient accumulation**: K micro-batches `+=` into one buffer, each scaled by `1/total`, equals one big batch (you verified it to `1e-7`). Zero the buffer once, before the loop; sync gradients only on the last micro-step.
- A **complete GPU training loop** is `forward → loss → backward → grad-norm → clip → AdamW`, the same control flow whether the model is one matmul or 124M parameters — and your linear model converged with the loss curve dropping and clipping active early.
- `gpt2_forward` / `gpt2_backward_and_reduce` route every layer to a CUDA kernel you've built; the classifier fuses softmax+CE+`dlogits` so the backward starts at the unembedding.
- The real per-step order and the real hyperparameters: betas `(0.9, 0.95)`, `max_norm = 1.0`, clip computed *before* AdamW, cast to BF16 *after*.

### What's next

**Chapter 20 — Multi-GPU with NCCL & ZeRO.** The final chapter. The one line you skipped above — `multi_gpu_async_reduce_gradient` — is where 8 GPUs synchronize. We'll *simulate* DDP all-reduce and ZeRO-1 reduce-scatter / all-gather on this single box (no cluster needed) and watch sharded optimizer state cut per-GPU memory.

When you're ready, say **"proceed to Chapter 20"**.
